# CS448 - Lab 5: Microphone Arrays

In this lab we will perform some simple microphone array processing. We will use the ```array.wav``` file. This is a recording from an 8-channel array. The microphones were placed at a distance of 0.1 meters from each other, and two simultaneous sounding sources were recorded. In the rest of this lab you will have to find where the sources are, and beamform so that you focus on each one separately.

In [1]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import scipy.signal as signal
# Make a sound player function that plays array "x" with a sample rate "rate", and labels it with "label"
def sound_player( x, rate=8000, label=''):
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))

## Part 1: Getting the Steering Vectors

In order to do any further processing on this array we will need to obtain a set of steering vectors. As you might recall the steering vectors encode the phase shift that each frequency undergoes between all the microphones of the array for a given source location.  Since we will be using a far-field model you will need to generate a steering vector for each frequency and source angle you want to check. We will assume that we will use 1024pt DFTs so that you need to generate steering vectors for 513 frequencies. You will also need to scan all angles from 0 to $\pi$. Since we won’t be making a continuous scan you can instead use 50 uniformly-spaced angles in that range.

Recall that the steering vector formula is:
$$v(k, \theta, m)=e^{-ⅈ\frac{m\cdot r \cdot cos(\theta)}{C} \frac{2\pi \cdot k \cdot R}{N}}$$

where $N$ is the size of the DFT and $k$ is a frequency index, $R$ is the sample rate, $C$ the speed of sound (use 345 m/s), $r$ is the distance between the mics, $m$ is the mic index (0 to 7 in this case) and $\theta$ is the angle to check. 


In [ ]:

x, sr=sf.read('array.wav')
print(x.shape)

# define constant variables 
# YOUR CODE HERE

# compute the steering vectors
# YOUR CODE HERE

(99615, 8)
(513, 50, 8)


## Part 2: Localization

Now that we have the steering vectors we can perform some localization. In order to find where the sources are we will make a beamformer that “focuses” at each of the 50 angles that we want to check and then returns the amount of acoustic energy that emanates from that point. Wherever there is a source we will see an energy bump. 

In order to perform the beamforming we will need to undo the phase shifts that are imposed on a sound from each angle. If we do so, for a signal that emanates from that angle we will appropriately phase shift the inputs so that they are perfectly synchronized over all 8 channels. If we have the desired source perfectly synchronized over all channels and we add them together we will boost that source by a factor of 8. If this synchrony is not present we will get a lesser boost.

In order to undo the phase shift we simply need to apply the inverse steering vectors on the input, which is nothing but the conjugate of the steering vectors you calculated above. 

Applying the steering vectors to the input signal is done in the frequency domain. Perform an STFT of each of the 8 input channels with 1024 frequencies, a hop size of 256 and a Hann window (I hope you have already finished our code from Lab 1!). This will result in a $F \times T \times 8$ tensor. In order to apply the necessary phase shift on each channel you need to multiply each STFT spectrum (i.e. each column of the STFT output) with the conjugate of the steering vector that corresponds to its microphone $m$ and the angle $\theta$ that you want to measure. For each angle you want to measure, do this over all the microphones and sum the resulting spectrograms from all the channels. 

*(Optional) You can implement this process by looping over the time frames and microphone channels, etc., but it might be quicker and more appropriate if you can define 3D arrays and use broadcasting. This is optional, but it will be fun to think in this way, because using nested loops in Python is a no-no. You can also use the einum function.*

Once you do that, you will have 50 spectrograms, each of which is the sum of the 8 time-aligned spectrograms. If this signal is loud enough, it means something. One way to measure the loudness of the signal is to do it in the time domain of course, but you can just calculate the magnitude values of the spectrogram and get their variance: $P(\theta)=\frac{1}{FT}\sum_{f,t} |Y_\theta(f,t)|^2$, where $Y_\theta(f,t)$ is the boosted (i.e., summed) spectrogram using the $\theta$ as the angle. This will be your response from angle $\theta$. Do this over all angles and plot the overall response. The peaks of the resulting plot will reveal to you where the sources are.


<!-- *For 4-hour credit:* Your processing of the STFTs should be a loop-less expression.  Hint: use `einsum` -->


In [80]:
# YOUR STFT and iSTFT functions here

# define the conjugate of the steering vectors for beamforming
# YOUR CODE HERE

# STFT for each channel (compute T from the function itself)
# YOUR CODE HERE

# v_conj is (F, Angles, Mics) = (513, 50, 8)
# YOUR CODE HERE

# Beamform all angles: the result is (F,T, Angles)
# YOUR CODE HERE

# Response per angle (compute the response for each angle by taking the mean power of the beamformed output across all time frames and frequency bins)
# YOUR CODE HERE

# plot the responses
# YOUR CODE HERE

## Part 3: Beamforming

Identify the angle of the two sources by looking at the peaks from the above result. Let’s call these $\theta_1$ and $\theta_2$. Now that you know where you want to focus the array, you can design two beamformers to focus on the two sources. The steering vectors that you need to use will be $v(\theta_1,:,:)$ and $v(\theta_2,:,:)$. Just as before you need to take each channel’s STFT, multiply each column with the conjugate of the steering vector that corresponds to all the channels and the selected angle to focus on, and then you simply add them all up. The resulting sum will the STFT of the focused output. Use your inverse STFT function to take this back to the time domain and verify that it indeed sounds better than any of the input channels.

*For 4-hour credit:* Your processing of the STFTs should be a loop-less expression.  Hint: use `einsum`


In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
